# ComfyUI + LTX 2.3 GGUF on Kaggle

**Backend:** kaggle-a  |  **Model:** LTX-2.3 Q2_K (12.4 GB)  |  **GPU:** T4 16GB

**Luồng:** Cài đặt → Download model → Start ComfyUI → Tunnel → Ghi URL lên Gist → Hermes dispatch

---

In [ ]:
# ===== CẤU HÌNH =====
BACKEND_NAME = "kaggle-a"
GIST_ID = "8da27f2e6e0d8809a043712cd90f9237"

# Phiên bản ComfyUI ổn định (đã kiểm chứng)
COMFY_COMMIT = "7fc3ccdcc2fb1f20c4b7dd4aca374db952fd66df"

print("✅ Config loaded")
print(f"   Backend: {BACKEND_NAME}")
print(f"   ComfyUI commit: {COMFY_COMMIT[:12]}...")

---
## Bước 1: Cài môi trường

In [ ]:
# Import thư viện + định nghĩa đường dẫn
import os, sys, subprocess, threading, time, json, requests
from datetime import datetime

HOME = '/kaggle/working'
COMFY = f'{HOME}/ComfyUI'
VENV = f'{HOME}/venv'
os.chdir(HOME)

print(f"HOME: {HOME}")
print(f"COMFY: {COMFY}")
print(f"VENV: {VENV}")

In [ ]:
# Tạo virtualenv với Python 3.10 (theo cách pogscafe 2202 votes)
import subprocess

if not os.path.exists(VENV):
    !pip install -q virtualenv
    !virtualenv {VENV} -p $(which python3.10)
    if not os.path.exists(f'{VENV}/bin/python3.10'):
        !cp /usr/bin/python3.10 {VENV}/bin/
    !ln -sf {VENV}/bin/python3.10 {VENV}/bin/python
    !ln -sf {VENV}/bin/python3.10 {VENV}/bin/python3

PYTHON = f'{VENV}/bin/python'
PIP = f'{VENV}/bin/pip'

# Kiểm tra
ok = os.path.exists(PYTHON)
print(f"✅ Python: {PYTHON}" if ok else f"❌ Python NOT FOUND at {PYTHON}")

In [ ]:
# Clone ComfyUI + ghim đúng phiên bản
if not os.path.exists(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
    print(f"  Cloned ComfyUI")

os.chdir(COMFY)
!git checkout {COMFY_COMMIT} 2>&1
print(f"  Pinned to commit {COMFY_COMMIT[:12]}")

# Cài dependencies
!{PIP} install -q -r requirements.txt 2>&1
print(f"  Dependencies installed")
print(f"✅ ComfyUI ready")

In [ ]:
# Clone custom nodes cho LTX 2.3
os.chdir(f'{COMFY}/custom_nodes')

for url, name in [
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git', 'ComfyUI-LTXVideo'),
    ('https://github.com/logtd/ComfyUI-LTXTricks.git', 'ComfyUI-LTXTricks'),
    ('https://github.com/city96/ComfyUI-GGUF.git', 'ComfyUI-GGUF'),
    ('https://github.com/ltdrdata/ComfyUI-Manager.git', 'ComfyUI-Manager'),
]:
    if not os.path.exists(name):
        !git clone {url} 2>&1
        print(f"  ✅ {name}")
    else:
        print(f"  ✔ {name} (already exists)")

print("")
print("✅ All custom nodes ready")

---
## Bước 2: Tải Models (mất ~15 phút)

In [ ]:
# Tạo symlink models -> /tmp (theo cách pogscafe)
!mkdir -p /tmp/models/{unet,clip,vae}

for d in ['unet', 'clip', 'vae']:
    src = f'{COMFY}/models/{d}'
    dst = f'/tmp/models/{d}'
    if os.path.islink(src) or os.path.exists(src):
        !rm -rf {src}
    !ln -sf {dst} {src}
    print(f"  Symlink: {d} -> /tmp/models/{d}")

print("✅ Symlinks ready")

In [ ]:
# Tải model chính: LTX-2.3 GGUF Q2_K (12.4 GB)
# CHỈ format này fit T4 16GB! FP8 (29GB) quá lớn
os.chdir('/tmp/models/unet')
MODEL_FILE = 'LTX-2.3-22B-distilled-1.1-Q2_K.gguf'
MODEL_URL = 'https://huggingface.co/QuantStack/LTX-2.3-GGUF/resolve/main/LTX-2.3-distilled-1.1/LTX-2.3-22B-distilled-1.1-Q2_K.gguf'

if not os.path.exists(MODEL_FILE):
    print("Downloading 12.4 GB model... (5-10 min)")
    !wget -c "{MODEL_URL}" -O "{MODEL_FILE}" 2>&1
    sz = os.path.getsize(MODEL_FILE) / 1e9
    print(f"   Done: {sz:.1f} GB")
else:
    sz = os.path.getsize(MODEL_FILE) / 1e9
    print(f"  Already exists: {sz:.1f} GB")

In [ ]:
# Tải text encoder + VAE (nhỏ, ~2 GB tổng)
for folder, filename, source_path in [
    ('clip', 'gemma-2b.safetensors', 'text_encoder/model.safetensors'),
    ('vae', 'ltx-vae.safetensors', 'vae/vae.safetensors'),
]:
    filepath = f'/tmp/models/{folder}/{filename}'
    if not os.path.exists(filepath):
        url = f'https://huggingface.co/Lightricks/LTX-2/resolve/main/{source_path}'
        print(f"  Downloading {filename}...")
        !wget -c "{url}" -O "{filepath}" 2>&1
        print(f"    OK ({os.path.getsize(filepath)/1e9:.1f} GB)")
    else:
        print(f"  ✔ {filename} (exists)")

print("✅ All models ready")

---
## Bước 3: Khởi động ComfyUI

In [ ]:
# Kill ComfyUI cũ nếu có
!pkill -f main.py 2>/dev/null
time.sleep(2)
print("  Clean up done")

# Start ComfyUI headless mode
os.chdir(COMFY)
comfy_proc = subprocess.Popen(
    [PYTHON, 'main.py', '--headless', '--port', '8188', '--listen', '127.0.0.1', '--highvram'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print(f"  Started ComfyUI (PID: {comfy_proc.pid})")

# Đợi ComfyUI sẵn sàng
for i in range(40):
    time.sleep(3)
    try:
        r = requests.get('http://127.0.0.1:8188/object_info', timeout=2)
        if r.status_code == 200:
            print(f"  ComfyUI API ready after {i*3}s")
            break
    except:
        pass
else:
    print("❌ ComfyUI FAILED to start!")

print("✅ ComfyUI running")

---
## Bước 4: Tunnel Pinggy

In [ ]:
# Tạo tunnel Pinggy ở background thread
TUNNEL_URL = None

def start_tunnel():
    global TUNNEL_URL
    cmd = ['ssh', '-p', '443', '-R0:localhost:8188', 'a.pinggy.io']
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in p.stdout:
        if 'https://' in line:
            i = line.find('https://')
            TUNNEL_URL = line[i:].strip().split()[0]
            with open(f'{HOME}/tunnel_url.txt', 'w') as f:
                f.write(TUNNEL_URL)
            print(f"TUNNEL URL: {TUNNEL_URL}")
            break

thread = threading.Thread(target=start_tunnel, daemon=True)
thread.start()
print("  Tunnel thread started")

# Chờ URL
time.sleep(15)
for i in range(30):
    if TUNNEL_URL:
        break
    try:
        TUNNEL_URL = open(f'{HOME}/tunnel_url.txt').read().strip()
    except:
        pass
    time.sleep(5)

if TUNNEL_URL:
    print(f"✅ Tunnel: {TUNNEL_URL}")
else:
    print("❌ Tunnel timeout - check Pinggy connection")

---
## Bước 5: Ghi URL lên GitHub Gist

In [ ]:
# Hàm ghi trạng thái lên Gist
def push_to_gist(url, status):
    # Đọc Gist hiện tại (giữ data của các backend khác)
    data = {}
    try:
        r = requests.get(f'https://api.github.com/gists/{GIST_ID}', timeout=5)
        if r.status_code == 200:
            c = r.json()['files']['kaggle_backends.json']['content']
            data = json.loads(c) if c.strip() else {}
    except:
        pass
    
    # Ghi thông tin backend này
    data[BACKEND_NAME] = {
        'url': url,
        'status': status,
        'updated': datetime.now().isoformat(),
        'capabilities': ['t2v', 'i2v', 'v2v'],
        'gpu': 't4'
    }
    
    # Push lên Gist
    r = requests.patch(f'https://api.github.com/gists/{GIST_ID}',
        json={'files': {'kaggle_backends.json': {'content': json.dumps(data, indent=2)}}},
        timeout=10
    )
    return r.status_code

if TUNNEL_URL:
    code = push_to_gist(TUNNEL_URL, 'online')
    if code == 200:
        print(f"✅ Gist updated! Backend: {BACKEND_NAME}")
        print(f"")
        print(f"   URL: {TUNNEL_URL}")
        print(f"")
        print(f"   Hermes can now dispatch jobs to:")
        print(f"   POST {TUNNEL_URL}/prompt")
    else:
        print(f"❌ Gist update failed (HTTP {code})")
else:
    print("❌ No tunnel URL available")

---
## Bước 6: Giữ kết nối

In [ ]:
# Heartbeat: cập nhật Gist mỗi 5 phút + giữ notebook sống
print(f"Backend {BACKEND_NAME} is ONLINE")
print(f"URL: {TUNNEL_URL}")
print(f"")
print(f"Waiting for jobs... Press Stop button to end")

try:
    counter = 0
    while True:
        time.sleep(300)
        counter += 1
        if TUNNEL_URL:
            push_to_gist(TUNNEL_URL, 'online')
        print(f"  [Heartbeat #{counter}] {datetime.now().isoformat()}")
except KeyboardInterrupt:
    if TUNNEL_URL:
        push_to_gist(TUNNEL_URL, 'offline')
    print("Backend stopped")